## M4 — Clustering K-Means
Segmentamos las 100 ubicaciones en perfiles para enriquecer la narrativa ante el jurado.

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('top100_ubicaciones.csv', encoding='utf-8-sig')

print(f"Ubicaciones cargadas: {df.shape[0]}")
print(f"Columnas: {df.columns.tolist()}")

Ubicaciones cargadas: 100
Columnas: ['CPRO', 'CMUN', 'NOMBRE', 'secciones', 'fofm_medio', 'fofm_max', 'vol_75plus_total', 'pct_75plus_media', 'ratio_viudez_media', 'cohorte_futura', 'renta_hogar', 'indice_cobertura', 'tiene_hospital', 'latitud', 'longitud', 'ranking_municipio', 'posicion_final']


### Paso 1 — Seleccionar variables y escalar

In [10]:
# Variables que definen el perfil de cada municipio
features = [
    'renta_hogar',
    'vol_75plus_total',
    'pct_75plus_media',
    'ratio_viudez_media',
    'cohorte_futura'
]

X = df[features].copy()

# Escalar — media 0, desviación estándar 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Variables escaladas:")
print(pd.DataFrame(X_scaled, columns=features).describe().round(2))

Variables escaladas:
       renta_hogar  vol_75plus_total  pct_75plus_media  ratio_viudez_media  \
count       100.00            100.00            100.00              100.00   
mean         -0.00              0.00              0.00               -0.00   
std           1.01              1.01              1.01                1.01   
min          -0.79             -0.73             -2.40               -2.12   
25%          -0.52             -0.64             -0.61               -0.54   
50%          -0.25             -0.40             -0.02               -0.01   
75%           0.07              0.20              0.66                0.61   
max           5.64              4.62              2.54                4.54   

       cohorte_futura  
count          100.00  
mean            -0.00  
std              1.01  
min             -0.75  
25%             -0.65  
50%             -0.38  
75%              0.19  
max              4.61  


### Paso 2 — Encontrar el número óptimo de clusters

In [11]:
resultados = []

for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil    = silhouette_score(X_scaled, labels)
    inercia = kmeans.inertia_
    resultados.append({'k': k, 'silhouette': round(sil, 4), 'inercia': round(inercia, 1)})

df_res = pd.DataFrame(resultados)
print(df_res.to_string(index=False))

 k  silhouette  inercia
 2      0.3580    349.3
 3      0.3983    264.8
 4      0.2999    206.1
 5      0.2570    172.3
 6      0.2943    140.5
 7      0.2930    120.9
 8      0.2674    106.0


### Paso 3 — Aplicar K-Means con 4 clusters

In [12]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Ver cuántos municipios hay en cada cluster
print("Municipios por cluster:")
print(df['cluster'].value_counts().sort_index())

# Ver las características medias de cada cluster
perfil = df.groupby('cluster')[features].mean().round(0)
print("\nPerfil medio por cluster:")
print(perfil.to_string())

Municipios por cluster:
cluster
0    54
1    11
2     5
3    30
Name: count, dtype: int64

Perfil medio por cluster:
         renta_hogar  vol_75plus_total  pct_75plus_media  ratio_viudez_media  cohorte_futura
cluster                                                                                     
0            30271.0            4489.0               9.0                 2.0          5334.0
1            30034.0           47896.0              11.0                 2.0         51877.0
2            67280.0            4462.0               7.0                 2.0          5806.0
3            31317.0           13528.0              13.0                 2.0         14045.0


### Paso 4 — Etiquetar los clusters con nombres estratégicos

In [13]:
nombres_cluster = {
    0: "Zona envejecida — solvencia ajustada",
    1: "Gran ciudad — mercado masivo",
    2: "Zona premium — máxima solvencia",
    3: "Municipio equilibrado — perfil estándar"
}

df['perfil'] = df['cluster'].map(nombres_cluster)

# Resumen por perfil
resumen = df.groupby('perfil').agg(
    municipios      = ('NOMBRE', 'count'),
    renta_media     = ('renta_hogar', 'mean'),
    vol_75plus      = ('vol_75plus_total', 'mean'),
    fofm_medio      = ('fofm_medio', 'mean')
).round(3)

print("=== PERFILES ESTRATÉGICOS ===")
print(resumen.to_string())

# Ver qué municipios hay en cada perfil
for perfil in nombres_cluster.values():
    municipios = df[df['perfil'] == perfil]['NOMBRE'].tolist()
    print(f"\n{perfil}:")
    print(", ".join(municipios[:10]))
    if len(municipios) > 10:
        print(f"  ... y {len(municipios)-10} más")

=== PERFILES ESTRATÉGICOS ===
                                         municipios  renta_media  vol_75plus  fofm_medio
perfil                                                                                  
Gran ciudad — mercado masivo                     11    30033.636   47895.818       0.277
Municipio equilibrado — perfil estándar          30    31317.333   13527.633       0.259
Zona envejecida — solvencia ajustada             54    30270.704    4488.778       0.257
Zona premium — máxima solvencia                   5    67280.000    4462.200       0.496

Zona envejecida — solvencia ajustada:
Esplugues de Llobregat, Sagunto/Sagunt, Alzira, Hondarribia, Sant Joan d'Alacant, Galdakao, Bormujos, Melilla, Santa Cruz de Bezana, Eivissa
  ... y 44 más

Gran ciudad — mercado masivo:
Valencia, Bilbao, Alicante/Alacant, Sevilla, Murcia, Vigo, Málaga, Córdoba, Zaragoza, Palmas de Gran Canaria, Las
  ... y 1 más

Zona premium — máxima solvencia:
Pozuelo de Alarcón, Boadilla del Monte, Majadaho

### Paso 5 — Guardar resultados del clustering

In [14]:
# Guardar top100 con cluster asignado
df.to_csv('top100_con_clusters.csv', index=False, encoding='utf-8-sig')

# Excel actualizado con la columna de perfil
df_muni = pd.read_csv('ranking_municipios.csv', encoding='utf-8-sig')
resumen_prov = pd.read_csv('resumen_provincias.csv', encoding='utf-8-sig')

with pd.ExcelWriter('resultados_akademia.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Top 100 ubicaciones', index=False)
    df_muni.head(500).to_excel(writer, sheet_name='Ranking municipios', index=False)
    resumen_prov.to_excel(writer, sheet_name='Resumen por provincia', index=False)
    
    # Nueva pestaña con resumen de perfiles
    resumen_clusters = df.groupby('perfil').agg(
        municipios  = ('NOMBRE', 'count'),
        renta_media = ('renta_hogar', 'mean'),
        vol_75plus  = ('vol_75plus_total', 'mean'),
        fofm_medio  = ('fofm_medio', 'mean')
    ).round(2).reset_index()
    resumen_clusters.to_excel(writer, sheet_name='Perfiles estratégicos', index=False)

print("Archivos guardados:")
print(f"  top100_con_clusters.csv → {len(df)} municipios con perfil asignado")
print(f"  resultados_akademia.xlsx → actualizado con pestaña de perfiles")

Archivos guardados:
  top100_con_clusters.csv → 100 municipios con perfil asignado
  resultados_akademia.xlsx → actualizado con pestaña de perfiles
